In [ ]:
!pip install --upgrade transformers objsize
!pip install -U "bitsandbytes>=0.46.1" accelerate
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerator bitsandbytes

  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.4 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.9.3 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=

In [ ]:
!pip install unsloth_zoo

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successfully uninstalled tokenizers-0.23.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0


In [ ]:
from huggingface_hub import login

login()

## Initializing the Model and the Tokenizer

In [ ]:
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported

# load the dataset
ds = load_dataset("gretelai/synthetic_text_to_sql")

# Load Model and Tokenizer using Unsloth
model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,  # Pre-quantized 4-bit Qwen2.5
    max_seq_length=max_seq_length,
    dtype=None,             # Auto-detects precision (FP16 for T4)
    load_in_4bit=True,      # 4-bit quantization for maximum speed and lowest VRAM
)

==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
size_bytes = model.get_memory_footprint()

in_mbs = size_bytes // (1024 ** 2)
print(f"{in_mbs} MBs")

1440 MBs


### Using Tokenizer

In [ ]:
x = tokenizer.encode('Hello, my beutiful model.')
print(x)

[9707, 11, 847, 387, 332, 4985, 1614, 13]


In [ ]:
print(tokenizer.decode(x))

Hello, my beutiful model.


In [ ]:
inputs =  tokenizer("Hello, World!", return_tensors='pt')
print(inputs['input_ids'])
print(inputs['attention_mask'])

tensor([[9707,   11, 4337,    0]])
tensor([[1, 1, 1, 1]])


### Using the Model

In [ ]:
messages = [{"role": "user", "content": "What is the capital of Egypt?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

input = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**input, max_new_tokens=50)

response = tokenizer.decode(output)
print(response)

Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of Egypt?<|im_end|>\n<|im_start|>assistant\nThe capital of Egypt is Cairo.\n\nCairo is the largest city in Egypt and serves as its political, economic, and cultural center. It is located on the Nile River in northeastern Egypt, about 150 kilometers south of the Mediterranean Sea.']


## Formating the data

In [ ]:
def format_prompt(example):
  question = example["sql_prompt"]
  db_schema = example['sql_context']
  answer = example['sql']

  chat = [
      {"role": "system", "content": "You are an SQL query generator given a question and a database schema, generate an SQL query."},
      {"role": "user", "content": f"Question: {question}\nDatabase Schema: {db_schema}\nSQL Query:"},
      {"role": "assistant", "content": f"{answer}"}
  ]
  formatted_chat = tokenizer.apply_chat_template(chat, tokenize=False)
  return {'text': formatted_chat}


### Checking a decent max amount of tokens to use

In [ ]:
import numpy as np

# Sample a chunk of the training set and tokenize the formatted text to see real lengths
sample = ds["train"].select(range(2000)).map(format_prompt)
lengths = [len(tokenizer(t)["input_ids"]) for t in sample["text"]]

lengths = np.array(lengths)
print(f"min={lengths.min()}, mean={lengths.mean():.0f}, "
      f"p90={np.percentile(lengths, 90):.0f}, p95={np.percentile(lengths, 95):.0f}, "
      f"p99={np.percentile(lengths, 99):.0f}, max={lengths.max()}")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

min=57, mean=180, p90=261, p95=293, p99=370, max=469


## Creating a custom Evaluation Metric

In [ ]:
import re

def sanitize_sql_for_sqlite(sql_str: str) -> str:
    if not sql_str:
        return ""

    # 1. Extract schema names (e.g., CREATE SCHEMA defense;)
    schemas = re.findall(r'(?i)CREATE\s+SCHEMA\s+(?:IF\s+NOT\s+EXISTS\s+)?([a-zA-Z0-9_]+)', sql_str)

    # 2. Strip CREATE and DROP SCHEMA statements
    sql = re.sub(r'(?i)(CREATE|DROP)\s+SCHEMA\s+[^;]+;', '', sql_str)

    # 3. Strip extracted schema prefixes (e.g., "defense.table_name" -> "table_name")
    for schema in schemas:
        sql = re.sub(rf'\b{re.escape(schema)}\.', '', sql)

    # 4. Fallback schema prefix cleanup for table targets (CREATE/INTO/FROM/JOIN)
    sql = re.sub(r'(?i)\b(TABLE|INTO|UPDATE|FROM|JOIN)\s+([a-zA-Z0-9_]+)\.', r'\1 ', sql)

    # 5. Remove Postgres typecasts (e.g., '2026-01-01'::DATE or col::TEXT)
    sql = re.sub(r'::\s*[a-zA-Z0-9_]+', '', sql)

    # 6. Translate keywords and data types
    sql = re.sub(r'\bILIKE\b', 'LIKE', sql, flags=re.IGNORECASE)
    sql = re.sub(r'\b(BIG)?SERIAL\b', 'INTEGER', sql, flags=re.IGNORECASE)
    sql = re.sub(r'\bTIMESTAMP(\s+WITH\s+TIME\s+ZONE)?\b', 'DATETIME', sql, flags=re.IGNORECASE)

    return sql

In [ ]:
import sqlite3
from collections import Counter

def validate_and_evaluate(schema_script: str, pred_sql: str, target_sql: str) -> dict:
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()

    status = {
        "target_executes": False,
        "pred_executes": False,
        "exact_data_match": False,
        "error": None,
    }

    try:
        cursor.executescript(sanitize_sql_for_sqlite(schema_script))

        try:
            cursor.execute(sanitize_sql_for_sqlite(target_sql))
            target_results = cursor.fetchall()
            status["target_executes"] = True
        except sqlite3.Error as e:
            status["error"] = f"target: {e}"
            return status

        try:
            cursor.execute(sanitize_sql_for_sqlite(pred_sql))
            pred_results = cursor.fetchall()
            status["pred_executes"] = True
        except sqlite3.Error as e:
            status["error"] = f"pred: {e}"
            return status

        # Counter compares row multiplicities, not just set membership --
        # a result missing/adding a duplicate row will now correctly count as a mismatch
        status["exact_data_match"] = Counter(pred_results) == Counter(target_results)

    except sqlite3.Error as e:
        status["error"] = f"schema: {e}"
    finally:
        conn.close()

    return status


the evaluation metric will be used in a different notebook "Evaluating Text2SQL Model", where I compare both the base and fine-tuned models.

## Initializing LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

# Add Fast LoRA Adapters
peft_model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,         # 0 is optimized for Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth", # Memory-saving optimization
    random_state=3000,
)

peft_model.print_trainable_parameters()

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


## Configuring the trainer

In [ ]:
# from transformers import TrainingArguments, Trainer
from trl import SFTTrainer, SFTConfig
# a more optimized trainer class

model_repo_name = "Qwen2.5-1.5B-Instruct-text2sql-lora"

# Create a robust, fast-training split
train_data = ds["train"].shuffle(seed=42).select(range(3500)).map(format_prompt)
test_data = ds["test"].select(range(350)).map(format_prompt)


# Configure Trainer
training_args = SFTConfig(
    output_dir=model_repo_name,
    dataset_text_field="text",
    max_length=1024,
    packing=False,

    # Speed & Memory Settings
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    num_train_epochs=2,

    # Automatic Hardware Floating-Point Detection
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="adamw_8bit",    # 8-bit optimizer
)

# Initialize Trainer & Train
trainer = SFTTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=test_data,
    args=training_args,
)

# start training
trainer.train()

peft_model.save_pretrained(model_repo_name)
tokenizer.save_pretrained(model_repo_name)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,500 | Num Epochs = 2 | Total steps = 110
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 9,232,384 of 1,552,946,688 (0.59% trained)


Epoch,Training Loss,Validation Loss
1,0.457638,0.448369
2,0.416670,0.443427


Unsloth: Restored added_tokens_decoder metadata in Qwen2.5-1.5B-Instruct-text2sql-lora/checkpoint-55/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in Qwen2.5-1.5B-Instruct-text2sql-lora/checkpoint-110/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in Qwen2.5-1.5B-Instruct-text2sql-lora/tokenizer_config.json.


('Qwen2.5-1.5B-Instruct-text2sql-lora/tokenizer_config.json',
 'Qwen2.5-1.5B-Instruct-text2sql-lora/chat_template.jinja',
 'Qwen2.5-1.5B-Instruct-text2sql-lora/tokenizer.json')

## Testing the model

In [ ]:
peft_model.eval()

test_case = ds["test"][250]

question = test_case["sql_prompt"]
db_schema = test_case['sql_context']
answer = test_case['sql']

chat = [
    {"role": "system", "content": "You are an SQL query generator given a question and a database schema, generate an SQL query."},
    {"role": "user", "content": f"Question: {question}\nDatabase Schema: {db_schema}\nSQL Query:"},
]

input = tokenizer(tokenizer.apply_chat_template(chat, tokenize=False), return_tensors="pt").to(peft_model.device)
input_len = input['input_ids'].shape[1]

output = peft_model.generate(**input, max_new_tokens=50)
decoded = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

print(f"Input: {chat[1]['content']}")
print(f"Output:{decoded}")
print(f"Answer:\n{answer}")

Both `max_new_tokens` (=50) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input: Question: Compare the number of economic diversification projects in two regions, one in Europe and one in Oceania, by showing the project type and the number of projects in each region.
Database Schema: CREATE TABLE economic_diversification (region VARCHAR(50), project_type VARCHAR(50), project_start_date DATE);
SQL Query:
Output:assistant
SELECT region, project_type, COUNT(*) as num_projects FROM economic_diversification GROUP BY region, project_type;
Answer:
SELECT 'Europe' as region, project_type, COUNT(*) as project_count FROM economic_diversification WHERE region = 'Europe' UNION ALL SELECT 'Oceania' as region, project_type, COUNT(*) as project_count FROM economic_diversification WHERE region = 'Oceania';


## Save the Model

In [ ]:
peft_model.push_to_hub(model_repo_name)
tokenizer.push_to_hub(model_repo_name)

README.md:   0%|          | 0.00/585 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.2kB / 37.0MB            

Saved model to https://huggingface.co/Qwen2.5-1.5B-Instruct-text2sql-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpwngxjvyn/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpwngxjvyn/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            